# CLAMP Dataset Explorer

Companion Jupyter Notebook for the **CLAMP Dataset** - **CLassified and Annotated Multimodal Postures**.

**Dataset authors and software contributors:**

Marta C. Mora, Noora Hamzah Shadahan Al-Owaidi, Bàrbara Pellicer-Coves, Jose V. García-Ortiz, and Joaquín Cerdá-Boluda

**Notebook version:**

1.0

**Date:**

June 2026

**Description:**

This notebook provides basic tools for loading, exploring, visualizing, and exporting data from the CLAMP dataset, a multimodal dataset containing surface electromyography (sEMG), accelerometer, and gyroscope recordings associated with hand postures and grasp-phase annotations.

The notebook can be used both locally and in Google Colab, provided that the dataset folders are placed in the same directory as this notebook.

## Dataset and citation information

**Dataset DOI:**  
https://doi.org/10.5281/zenodo.20555985

**Software repository:**  
CLAMP-tools  
`https://github.com/ximocerda/CLAMP-tools`

**Dataset repository:**  
Zenodo

**Dataset license:**  
Creative Commons Attribution 4.0 International License (CC BY 4.0)

**Software license:**  
MIT License

**Citation:**

If you use this dataset or the accompanying software tools, please cite the CLAMP dataset and the associated article.

Dataset DOI: https://doi.org/10.5281/zenodo.20555985

## Requirements and imports

This notebook uses standard Python scientific libraries for data loading, visualization, interactive exploration, and export utilities.

If you are running the notebook locally, you can install the required dependencies using:

```bash
pip install -r requirements.txt
```

If you are running the notebook in Google Colab, most dependencies are usually available by default. If any package is missing, install it before running the following cells.


In [ ]:
from pathlib import Path
import json
import shutil
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError as exc:
    raise ImportError(
        "ipywidgets is required to run the interactive components of this notebook. "
        "Please install it using: pip install ipywidgets"
    ) from exc

warnings.filterwarnings("ignore", category=FutureWarning)

print("Imports completed successfully.")

## Dataset folder setup
The folders ```raw_data/``` and ```processed_data/``` should be obtained from the CLAMP dataset repository in Zenodo and placed in the same directory as this notebook. These folders are distributed through Zenodo as two compressed files:

- `raw_data.zip`
- `processed_data.zip`

Please download both files from Zenodo and place them in the same directory as `CLAMP_dataset_explorer.ipynb`.

Before running the notebook, the folder may look like this:

```text
CLAMP_dataset_explorer.ipynb
raw_data.zip
processed_data.zip
```

After extracting the files, the expected structure is:

```text
CLAMP_dataset_explorer.ipynb
raw_data/
processed_data/
```

The next cell checks whether the folders already exist. If they do not exist but the ZIP files are present, the cell will extract them automatically.

The notebook mainly uses the ```processed_data/``` folder. The ```raw_data/``` folder is included for completeness and for users interested in accessing the original MATLAB recordings.

In [ ]:
BASE_DIR = Path.cwd()

RAW_DATA_DIR = BASE_DIR / "raw_data"
PROCESSED_DATA_DIR = BASE_DIR / "processed_data"

RAW_DATA_ZIP = BASE_DIR / "raw_data.zip"
PROCESSED_DATA_ZIP = BASE_DIR / "processed_data.zip"


def extract_zip_if_needed(zip_path: Path, expected_dir: Path) -> None:
    """
    Extract a ZIP file only if the expected output folder does not already exist.
    """
    if expected_dir.exists():
        print(f"Found existing folder: {expected_dir.name}/")
        return

    if not zip_path.exists():
        print(f"ZIP file not found: {zip_path.name}")
        return

    print(f"Extracting {zip_path.name}...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(BASE_DIR)

    if expected_dir.exists():
        print(f"Extraction completed: {expected_dir.name}/")
    else:
        raise FileNotFoundError(
            f"{zip_path.name} was extracted, but {expected_dir.name}/ was not found. "
            "Please check the ZIP file structure."
        )


extract_zip_if_needed(RAW_DATA_ZIP, RAW_DATA_DIR)
extract_zip_if_needed(PROCESSED_DATA_ZIP, PROCESSED_DATA_DIR)

if not PROCESSED_DATA_DIR.exists():
    raise FileNotFoundError(
        "processed_data/ folder not found. Please download processed_data.zip from Zenodo "
        "and place it in the same directory as this notebook."
    )

print("\nDataset folder setup completed.")
print(f"Base directory: {BASE_DIR}")
print(f"Raw data folder: {RAW_DATA_DIR if RAW_DATA_DIR.exists() else 'not available'}")
print(f"Processed data folder: {PROCESSED_DATA_DIR}")

## Quick dataset check

This optional cell performs a quick check to verify that the processed dataset can be loaded correctly. It loads one posture-level group and one subject–posture group and displays basic information about the signal matrix, trials, and metadata.

In [ ]:
# Quick dataset check

# Example 1: accelerometer data for posture P01 across all subjects
example_posture_path = PROCESSED_DATA_DIR / "acc" / "by_posture" / "P01"

X_posture = np.load(example_posture_path / "matrix.npy")
trials_posture = pd.read_csv(example_posture_path / "trials.csv")

with open(example_posture_path / "metadata.json", "r", encoding="utf-8") as f:
    meta_posture = json.load(f)

print("Posture-level example")
print("---------------------")
print(f"Path: {example_posture_path}")
print(f"Signal matrix shape: {X_posture.shape}")
print(f"Posture name: {meta_posture.get('posture_name')}")
print(f"Subjects included: {meta_posture.get('subjects_included')[:3]} ...")
print(f"Number of trials: {len(trials_posture)}")

print("\n")

# Example 2: accelerometer data for subject S01 and posture P01
example_subject_path = PROCESSED_DATA_DIR / "acc" / "by_subject_posture" / "S01" / "P01"

X_subject = np.load(example_subject_path / "matrix.npy")
trials_subject = pd.read_csv(example_subject_path / "trials.csv")

with open(example_subject_path / "metadata.json", "r", encoding="utf-8") as f:
    meta_subject = json.load(f)

print("Subject–posture example")
print("-----------------------")
print(f"Path: {example_subject_path}")
print(f"Signal matrix shape: {X_subject.shape}")
print(f"Posture name: {meta_subject.get('posture_name')}")
print(f"Subject: {meta_subject.get('subject_id')}")
print(f"Age: {meta_subject.get('age')}")
print(f"Gender: {meta_subject.get('gender')}")
print(f"HB: {meta_subject.get('hb')}")
print(f"HL: {meta_subject.get('hl')}")
print(f"Number of trials: {len(trials_subject)}")


## Basic loading function

The following function loads one dataset group from the `processed_data/` directory.

A group can be loaded either by posture across all subjects (`by_posture`) or by subject and posture (`by_subject_posture`). Each group contains:

- `matrix.npy`: signal matrix
- `trials.csv`: trial-level information
- `index.csv`: sample-level annotations
- `metadata.json`: group metadata

In [ ]:
def _format_id(value, prefix):
    """
    Convert numeric or string identifiers to dataset folder format.

    Examples
    --------
    _format_id(1, "P") -> "P01"
    _format_id("P01", "P") -> "P01"
    _format_id(3, "S") -> "S03"
    """
    if isinstance(value, int):
        return f"{prefix}{value:02d}"

    if isinstance(value, str):
        value = value.strip().upper()
        if value.startswith(prefix):
            return value
        if value.isdigit():
            return f"{prefix}{int(value):02d}"

    raise ValueError(f"Invalid identifier: {value}. Expected an integer or a string such as '{prefix}01'.")


def load_dataset(
    modality,
    posture,
    subject=None,
    grouping="by_subject_posture",
    root=None
):
    """
    Load one group from the processed CLAMP dataset.

    Parameters
    ----------
    modality : str
        Signal modality. Expected values are "emg", "acc", or "gyro".

    posture : int or str
        Posture identifier. Examples: 1, "1", or "P01".

    subject : int or str, optional
        Subject identifier. Required when grouping is "by_subject_posture".
        Examples: 1, "1", or "S01".

    grouping : str
        Dataset grouping strategy. Expected values are:
        - "by_subject_posture": data from one subject and one posture.
        - "by_posture": data from all subjects grouped by posture.

    root : str or pathlib.Path, optional
        Path to the processed dataset folder. If None, PROCESSED_DATA_DIR is used.

    Returns
    -------
    X : numpy.ndarray
        Signal matrix.

    trials : pandas.DataFrame
        Trial-level information.

    index : pandas.DataFrame
        Sample-level annotations.

    metadata : dict
        Group metadata.
    """
    if root is None:
        root = PROCESSED_DATA_DIR

    root = Path(root)

    modality = modality.lower().strip()
    valid_modalities = {"emg", "acc", "gyro"}

    if modality not in valid_modalities:
        raise ValueError(
            f"Invalid modality: {modality}. Expected one of {sorted(valid_modalities)}."
        )

    valid_groupings = {"by_subject_posture", "by_posture"}

    if grouping not in valid_groupings:
        raise ValueError(
            f"Invalid grouping: {grouping}. Expected one of {sorted(valid_groupings)}."
        )

    posture_id = _format_id(posture, "P")

    if grouping == "by_posture":
        group_path = root / modality / "by_posture" / posture_id

    else:
        if subject is None:
            raise ValueError(
                "subject must be specified when grouping='by_subject_posture'."
            )

        subject_id = _format_id(subject, "S")
        group_path = root / modality / "by_subject_posture" / subject_id / posture_id

    if not group_path.exists():
        raise FileNotFoundError(
            f"Dataset group not found: {group_path}"
        )

    required_files = {
        "matrix": group_path / "matrix.npy",
        "trials": group_path / "trials.csv",
        "index": group_path / "index.csv",
        "metadata": group_path / "metadata.json",
    }

    missing_files = [
        name for name, path in required_files.items()
        if not path.exists()
    ]

    if missing_files:
        raise FileNotFoundError(
            f"Missing files in {group_path}: {missing_files}"
        )

    X = np.load(required_files["matrix"])
    trials = pd.read_csv(required_files["trials"])
    index = pd.read_csv(required_files["index"])

    with open(required_files["metadata"], "r", encoding="utf-8") as f:
        metadata = json.load(f)

    return X, trials, index, metadata

## Example: loading one dataset group

The following example loads accelerometer data for subject `S01` and posture `P01`.

In [ ]:
X, trials, index, metadata = load_dataset(
    modality="acc",
    posture="P01",
    subject="S01",
    grouping="by_subject_posture"
)

print("Signal matrix shape:", X.shape)
print("Number of trials:", len(trials))
print("Number of indexed samples:", len(index))
print("Posture name:", metadata.get("posture_name"))
print("Subject:", metadata.get("subject_id"))

## Example usage

The following examples show how to load data using the two organization modes provided in the dataset:

- `by_posture`: all subjects grouped by posture.
- `by_subject_posture`: one specific subject and posture.

Only a short summary is printed in order to keep the notebook readable.

In [ ]:
# Example 1: accelerometer data for posture P01 across all subjects

X, trials, index, metadata = load_dataset(
    modality="acc",
    posture="P01",
    grouping="by_posture"
)

print("Example 1: acc / by_posture / P01")
print("----------------------------------")
print("Signal matrix shape:", X.shape)
print("Number of trials:", len(trials))
print("Number of indexed samples:", len(index))
print("Posture name:", metadata.get("posture_name"))
print("Subjects included:", metadata.get("subjects_included"))

print("\n")

# Example 2: accelerometer data for subject S01 and posture P01

X, trials, index, metadata = load_dataset(
    modality="acc",
    posture="P01",
    subject="S01",
    grouping="by_subject_posture"
)

print("Example 2: acc / by_subject_posture / S01 / P01")
print("------------------------------------------------")
print("Signal matrix shape:", X.shape)
print("Number of trials:", len(trials))
print("Number of indexed samples:", len(index))
print("Posture name:", metadata.get("posture_name"))
print("Subject:", metadata.get("subject_id"))
print("Age:", metadata.get("age"))
print("Gender:", metadata.get("gender"))
print("HB:", metadata.get("hb"))
print("HL:", metadata.get("hl"))

To load sEMG or gyroscope data, change the `modality` argument to `"emg"` or `"gyro"`.

## Interactive dataset explorer

The following interface allows users to select the signal modality, dataset organization, subject, posture, grasp-phase filter, and starting sample for visualization.

After selecting the desired options, click **Load and display dataset** to load the selected group and visualize the corresponding signal segment.

In [ ]:
CURRENT_DATA = {}


def _as_boolean_mask(series):
    """
    Convert a pandas Series into a boolean mask.
    This function supports boolean, numeric, and string-based annotations.
    """
    if pd.api.types.is_bool_dtype(series):
        return series.to_numpy()

    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(int).astype(bool).to_numpy()

    return series.astype(str).str.lower().isin(["true", "1", "yes"]).to_numpy()


def explore(modality, grouping, subject, posture, only_grasp, start):
    """
    Explore one selected group from the processed CLAMP dataset.

    Parameters
    ----------
    modality : str
        Signal modality: "emg", "acc", or "gyro".

    grouping : str
        Dataset organization mode: "by_posture" or "by_subject_posture".

    subject : str
        Subject identifier, for example "S01". Used only when grouping is
        "by_subject_posture".

    posture : str
        Posture identifier, for example "P01".

    only_grasp : bool
        If True, only samples with in_grasp annotation are displayed.

    start : int
        Starting sample used for signal visualization.
    """
    # Load selected dataset group
    if grouping == "by_posture":
        X, trials, index, metadata = load_dataset(
            modality=modality,
            posture=posture,
            grouping="by_posture"
        )
    else:
        X, trials, index, metadata = load_dataset(
            modality=modality,
            posture=posture,
            subject=subject,
            grouping="by_subject_posture"
        )

    # Keep original data for traceability
    X_original = X
    index_original = index

    # Optional in_grasp filtering for display
    is_filtered = False

    if only_grasp:
        if "in_grasp" not in index.columns:
            raise ValueError(
                "The selected index.csv file does not contain an 'in_grasp' column."
            )

        mask = _as_boolean_mask(index["in_grasp"])
        X = X[mask]
        index = index.loc[mask].reset_index(drop=True)
        is_filtered = True

    # Store current selection for possible later use
    CURRENT_DATA.clear()
    CURRENT_DATA.update({
        "X": X,
        "trials": trials,
        "index": index,
        "metadata": metadata,
        "X_original": X_original,
        "index_original": index_original,
        "modality": modality,
        "grouping": grouping,
        "subject": subject if grouping == "by_subject_posture" else None,
        "posture": posture,
        "only_grasp": only_grasp,
        "is_filtered": is_filtered,
    })

    # Display basic information
    print("Selected dataset group")
    print("----------------------")
    print(f"Modality: {modality}")
    print(f"Grouping: {grouping}")

    if grouping == "by_subject_posture":
        print(f"Subject: {metadata.get('subject_id', subject)}")

    print(f"Posture: {metadata.get('posture_name', posture)}")
    print(f"Signal matrix shape: {X.shape}")
    print(f"Number of trials: {len(trials)}")
    print(f"Number of indexed samples: {len(index)}")

    if "subjects_included" in metadata:
        print(f"Subjects included: {metadata['subjects_included']}")

    if only_grasp:
        print("Display filter: only in_grasp samples")

    # Preview tables
    print("\nTrials preview")
    display(trials.head())

    print("\nIndex preview")
    display(index.head())

    # Signal visualization
    if len(X) == 0:
        raise ValueError("No samples available for the selected configuration.")

    window_size = min(1000, len(X))
    start = int(start)
    start = max(0, min(start, max(0, len(X) - window_size)))

    segment = X[start:start + window_size, :]

    # Dynamic vertical offset for visualization
    if segment.shape[1] > 1:
        signal_range = np.nanmax(segment) - np.nanmin(segment)
        offset = signal_range * 1.2 if signal_range > 0 else 1
    else:
        offset = 0

    plt.figure(figsize=(12, 6))

    for ch in range(segment.shape[1]):
        plt.plot(segment[:, ch] + ch * offset, label=f"Ch {ch + 1}")

    title = f"{modality} - {grouping} - {posture} | Samples {start}:{start + window_size}"

    if grouping == "by_subject_posture":
        title = f"{modality} - {subject} - {posture} | Samples {start}:{start + window_size}"

    if only_grasp:
        title += " | in_grasp only"

    plt.title(title)
    plt.xlabel("Samples")
    plt.ylabel("Signal amplitude, vertically offset for display")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive widgets

modality_w = widgets.Dropdown(
    options=["emg", "acc", "gyro"],
    value="emg",
    description="Modality:",
    style={"description_width": "initial"}
)

grouping_w = widgets.ToggleButtons(
    options=["by_posture", "by_subject_posture"],
    value="by_subject_posture",
    description="Grouping:",
    style={"description_width": "initial"}
)

subject_w = widgets.Dropdown(
    options=[f"S{i:02d}" for i in range(1, 11)],
    value="S01",
    description="Subject:",
    style={"description_width": "initial"}
)

posture_w = widgets.Dropdown(
    options=[f"P{i:02d}" for i in range(1, 10)],
    value="P01",
    description="Posture:",
    style={"description_width": "initial"}
)

only_grasp_w = widgets.Checkbox(
    value=False,
    description="Only in_grasp samples",
    indent=False
)

start_w = widgets.IntSlider(
    value=0,
    min=0,
    max=250000,
    step=100,
    description="Start sample:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px")
)

run_button = widgets.Button(
    description="Load and display dataset",
    button_style="primary",
    tooltip="Load selected dataset group and display signal segment"
)

output = widgets.Output()


def update_subject_visibility(change=None):
    """
    Disable the subject selector when data are grouped by posture across subjects.
    """
    subject_w.disabled = grouping_w.value == "by_posture"


def run_explorer(button):
    """
    Run the explorer using the current widget values.
    """
    with output:
        clear_output(wait=True)

        explore(
            modality=modality_w.value,
            grouping=grouping_w.value,
            subject=subject_w.value,
            posture=posture_w.value,
            only_grasp=only_grasp_w.value,
            start=start_w.value
        )


grouping_w.observe(update_subject_visibility, names="value")
run_button.on_click(run_explorer)

update_subject_visibility()

display(
    modality_w,
    grouping_w,
    subject_w,
    posture_w,
    only_grasp_w,
    start_w,
    run_button,
    output
)

## Export selected dataset group

The following tool exports the currently selected dataset group to a local `exports/` folder.

The export function copies the complete selected group, including:

- `matrix.npy`
- `trials.csv`
- `index.csv`
- `metadata.json`
- `bundle.mat`, if available

The `Only in_grasp samples` option is used only for visualization. Exported data always correspond to the complete selected group in order to preserve consistency between the signal matrix, trial descriptors, sample-level annotations, and metadata.

The export tool does not generate a filtered or modified dataset. It copies the complete currently selected dataset group to an `exports/` folder, preserving the original relationship between signal matrices, annotations, trial descriptors, and metadata.

In [ ]:
def _safe_folder_name(name):
    """
    Convert a user-provided folder name into a safe folder name.
    """
    name = str(name).strip()

    if not name:
        return ""

    return "".join(
        char if char.isalnum() or char in "-_." else "_"
        for char in name
    ).strip("_")


def _get_current_group_path():
    """
    Reconstruct the path of the currently selected dataset group.
    """
    if not CURRENT_DATA:
        raise RuntimeError(
            "No dataset group has been selected yet. "
            "Please run the interactive explorer before exporting."
        )

    modality = CURRENT_DATA["modality"]
    grouping = CURRENT_DATA["grouping"]
    posture = CURRENT_DATA["posture"]
    subject = CURRENT_DATA.get("subject")

    if grouping == "by_posture":
        return PROCESSED_DATA_DIR / modality / "by_posture" / posture

    if grouping == "by_subject_posture":
        return PROCESSED_DATA_DIR / modality / "by_subject_posture" / subject / posture

    raise ValueError(f"Unknown grouping mode: {grouping}")


def export_current_selection(export_folder_name=""):
    """
    Export the currently selected complete dataset group.

    The export operation copies the original files from the processed dataset
    instead of exporting a temporary filtered visualization subset.
    """
    source_dir = _get_current_group_path()

    if not source_dir.exists():
        raise FileNotFoundError(f"Selected dataset group not found: {source_dir}")

    modality = CURRENT_DATA["modality"]
    grouping = CURRENT_DATA["grouping"]
    posture = CURRENT_DATA["posture"]
    subject = CURRENT_DATA.get("subject")

    safe_name = _safe_folder_name(export_folder_name)

    if not safe_name:
        if grouping == "by_posture":
            safe_name = f"{modality}_{grouping}_{posture}"
        else:
            safe_name = f"{modality}_{grouping}_{subject}_{posture}"

    export_root = BASE_DIR / "exports"
    export_dir = export_root / safe_name
    export_dir.mkdir(parents=True, exist_ok=True)

    files_to_copy = [
        "matrix.npy",
        "trials.csv",
        "index.csv",
        "metadata.json",
        "bundle.mat",
    ]

    copied_files = []

    for filename in files_to_copy:
        source_file = source_dir / filename

        if source_file.exists():
            shutil.copy2(source_file, export_dir / filename)
            copied_files.append(filename)

    print("Export completed successfully.")
    print(f"Source folder: {source_dir}")
    print(f"Export folder: {export_dir}")
    print("Copied files:", copied_files)

    if CURRENT_DATA.get("only_grasp"):
        print(
            "\nNote: the current visualization used the in_grasp filter, "
            "but the exported files correspond to the complete selected group."
        )

    return export_dir

In [ ]:
# Export widgets

export_name_w = widgets.Text(
    value="",
    placeholder="optional folder name",
    description="Export folder:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

export_button = widgets.Button(
    description="Export selected group",
    button_style="success",
    tooltip="Export the currently selected complete dataset group"
)

export_output = widgets.Output()


def run_export(button):
    """
    Run export using the current widget value.
    """
    with export_output:
        clear_output(wait=True)

        try:
            export_current_selection(export_name_w.value)
        except Exception as exc:
            print(f"Export failed: {exc}")


export_button.on_click(run_export)

display(export_name_w, export_button, export_output)

## Further analysis

This notebook provides basic tools for loading, exploring, visualizing, and exporting selected groups from the CLAMP dataset.

It is intentionally limited to dataset access and inspection. Users can build their own preprocessing, feature extraction, statistical analysis, or machine learning workflows from the loaded signal matrices and annotation files.

The `index.csv` file includes sample-level annotations such as `in_grasp` and `phase_code`, which can be used to select specific temporal segments or grasp phases according to the requirements of each application.